# Genie360 — 01: Query History Exploration

Interactive analysis notebook for exploring Genie-generated queries:
- Query `system.query.history` filtered by `genie_space_id`
- Visualize top offenders by cost and duration
- Cost distribution analysis
- Anti-pattern frequency heatmap
- Used during development and onboarding

In [1]:
from __future__ import annotations

import os
import re
import json
from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter
from typing import Any

import yaml
import pandas as pd
from dotenv import load_dotenv

def _find_file(filename: str) -> Path:
    """Search cwd, parent, and grandparent for *filename*."""
    for directory in [Path(os.getcwd()), Path(os.getcwd()).parent, Path(os.getcwd()).parent.parent]:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"{filename} not found in cwd or ancestor directories")


def load_yaml_config(config_path: str | Path | None = None) -> dict[str, Any]:
    """Load application config from *config.yml* and credentials from *.env*.

    Non-secret tunables (catalog, schema, tables, etc.) live in config.yml.
    Credentials (DATABRICKS_HOST / DATABRICKS_TOKEN) are loaded from .env.
    """
    if config_path is None:
        config_path = _find_file("config.yml")
    config_path = Path(config_path)

    with open(config_path) as fh:
        cfg = yaml.safe_load(fh)
    print(f"✅ Loaded config from {config_path}")

    return cfg

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from genie360.pipeline import load_genie360_config
from genie360.modules.query_history_ingestion import (
    get_genie_queries,
    normalize_query_record,
    deduplicate_queries,
    enrich_query_metadata,
    calculate_query_cost_estimate,
)

# config = load_genie360_config(project_root / "genie360" / "config.yml")

config = load_yaml_config()

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

print(f"Connected — Spark {spark.version}")

✅ Loaded config from /Users/sourav.banerjee/Documents/Codebases/Databricks Projects/AutoGenie/genie360/config.yml
Connected — Spark 4.0.0


## 1. Discover Genie Spaces

In [3]:
# Find all active Genie Spaces from query history
spaces_df = spark.sql("""
    SELECT
        query_source.genie_space_id AS space_id,
        COUNT(*) AS query_count,
        AVG(total_duration_ms) AS avg_duration_ms,
        SUM(read_bytes) / 1e9 AS total_gb_scanned,
        MIN(start_time) AS first_query,
        MAX(start_time) AS last_query
    FROM system.query.history
    WHERE start_time >= CURRENT_TIMESTAMP - INTERVAL 30 DAYS
      AND query_source.genie_space_id IS NOT NULL
      AND statement_type = 'SELECT'
    GROUP BY query_source.genie_space_id
    ORDER BY query_count DESC
""")

display(spaces_df)

,space_id,query_count,avg_duration_ms,total_gb_scanned,first_query,last_query
0,01f10e753b941c87ac6dd970a5c360f1,13896,2770.158967,0.191996,2026-02-20 16:00:20.584,2026-02-21 05:37:55.276
1,01f10130ec2915e8a8022a4f4f4d4d96,1726,1262.809386,5.587010,2026-02-03 18:48:27.348,2026-02-24 19:38:54.902
2,01f07d0e09a91904b1269f1de526d606,961,1369.240375,0.242886,2026-01-27 16:41:16.522,2026-02-24 05:11:25.764
3,01f10d3981781837ad28ee18953f43f8,811,2769.882861,0.069926,2026-02-19 02:20:31.600,2026-02-20 15:47:05.686
4,01f10d4664e216f2936c2d8b8c2e1bdb,652,657.670245,0.010497,2026-02-19 03:52:27.935,2026-02-24 18:08:08.207
5,01f107246d9a1696af9a846cb49ef8b8,628,420.587580,3.423877,2026-02-11 09:38:55.020,2026-02-18 17:04:21.271
6,01f105cf03cf1820b2dfb3903934bac9,573,3997.804538,1.959763,2026-02-09 15:50:13.894,2026-02-18 14:10:12.922
7,01f10b0606021b5b8885474df2da985c,560,3508.816071,0.182883,2026-02-16 07:06:52.875,2026-02-24 09:10:35.199
8,01f109283150185cb6cacdb55c4a50ba,526,3030.355513,0.155157,2026-02-13 22:06:26.892,2026-02-17 09:31:29.958
9,01ef11fa193b1983b8953b7119ff3b66,457,633.938731,0.023896,2026-01-26 14:02:53.825,2026-02-24 05:13:34.678


In [3]:
# Select a space to analyze (change this to your target space)
# TARGET_SPACE_ID = spaces_df.collect()[0]["space_id"] if spaces_df.count() > 0 else None

TARGET_SPACE_ID = "01f10e753b941c87ac6dd970a5c360f1"
print(f"Analyzing space: {TARGET_SPACE_ID}")

Analyzing space: 01f10e753b941c87ac6dd970a5c360f1


## 2. Ingest & Enrich Queries

In [4]:
raw_queries = get_genie_queries(spark, TARGET_SPACE_ID, lookback_days=30, limit=500)
print(f"Retrieved {len(raw_queries)} raw queries")

records = [normalize_query_record(r) for r in raw_queries]
deduped = deduplicate_queries(records)
enriched = [enrich_query_metadata(r) for r in deduped]

print(f"After deduplication: {len(deduped)} unique queries")
print(f"Enriched: {len(enriched)} queries with cost/tier metadata")

Retrieved 500 raw queries
After deduplication: 35 unique queries
Enriched: 35 queries with cost/tier metadata


## 3. Top Offenders — Slowest Queries

In [6]:
import pandas as pd

df = pd.DataFrame([r.model_dump() for r in enriched])

top_slow = df.nlargest(10, "total_duration_ms")[
    ["query_id", "total_duration_ms", "read_bytes", "estimated_cost_usd",
     "duration_tier", "volume_tier", "from_result_cache"]
]
print("\n🐢 Top 10 Slowest Genie Queries:")
top_slow


🐢 Top 10 Slowest Genie Queries:


,query_id,total_duration_ms,read_bytes,estimated_cost_usd,duration_tier,volume_tier,from_result_cache
24,01f10ee3-087b-1bb7-b4e8-25a64dda8a74,12699,0,0.0247,DurationTier.SLOW,VolumeTier.SMALL,True
7,01f10ee3-083d-1fc0-bfda-b66c4cab6fdb,12059,39708,0.0234,DurationTier.SLOW,VolumeTier.SMALL,False
3,01f10ee3-082e-1789-b6f7-42a334e33d65,11824,19854,0.0230,DurationTier.SLOW,VolumeTier.SMALL,False
26,01f10ea7-d997-1ccb-a138-b0431556672a,11384,4365,0.0221,DurationTier.SLOW,VolumeTier.SMALL,False
27,01f10ec1-8278-1c33-a05c-eb8488d3a608,10809,73696,0.0210,DurationTier.SLOW,VolumeTier.SMALL,False
17,01f10eb0-dd8b-1bfb-83f0-f4e4848fde66,10785,26524,0.0210,DurationTier.SLOW,VolumeTier.SMALL,False
28,01f10ea8-4fd0-149b-8212-a418e0faae82,10252,4365,0.0199,DurationTier.SLOW,VolumeTier.SMALL,False
10,01f10ee4-bda8-17e4-a9b0-d908f1f2c0da,9842,19854,0.0191,DurationTier.MODERATE,VolumeTier.SMALL,False
30,01f10ec8-82ed-147d-bfd5-661038e5215a,9475,73696,0.0184,DurationTier.MODERATE,VolumeTier.SMALL,False
23,01f10ee6-e871-124f-aa2f-04f5420c86bc,9286,0,0.0181,DurationTier.MODERATE,VolumeTier.SMALL,True


## 4. Cost Distribution

In [7]:
print("\n💰 Cost Distribution:")
print(f"  Total estimated cost: ${df['estimated_cost_usd'].sum():.2f}")
print(f"  Average per query:    ${df['estimated_cost_usd'].mean():.4f}")
print(f"  P50 cost:             ${df['estimated_cost_usd'].median():.4f}")
print(f"  P90 cost:             ${df['estimated_cost_usd'].quantile(0.9):.4f}")
print(f"  Max single query:     ${df['estimated_cost_usd'].max():.4f}")

print("\n⏱ Duration Distribution:")
print(df['duration_tier'].value_counts().to_string())

print("\n📦 Volume Distribution:")
print(df['volume_tier'].value_counts().to_string())


💰 Cost Distribution:
  Total estimated cost: $0.61
  Average per query:    $0.0175
  P50 cost:             $0.0163
  P90 cost:             $0.0217
  Max single query:     $0.0247

⏱ Duration Distribution:
duration_tier
DurationTier.MODERATE    28
DurationTier.SLOW         7

📦 Volume Distribution:
volume_tier
VolumeTier.SMALL    35


## 5. Anti-Pattern Frequency Analysis

In [8]:
from genie360.modules.anti_pattern_detection import run_anti_pattern_suite
from collections import Counter

pattern_counter = Counter()
severity_counter = Counter()
queries_with_issues = 0

for record in enriched:
    report = run_anti_pattern_suite(record.statement_text)
    if report.total_issues > 0:
        queries_with_issues += 1
        for p in report.patterns:
            pattern_counter[p.pattern_type] += 1
            severity_counter[p.severity.value] += 1

print(f"\n🔍 Anti-Pattern Summary:")
print(f"  Queries with issues: {queries_with_issues}/{len(enriched)} "
      f"({queries_with_issues/len(enriched)*100:.0f}%)" if enriched else "No queries")

print("\n  By Pattern Type:")
for pattern, count in pattern_counter.most_common():
    print(f"    {pattern}: {count}")

print("\n  By Severity:")
for severity, count in severity_counter.most_common():
    print(f"    {severity}: {count}")


🔍 Anti-Pattern Summary:
  Queries with issues: 7/35 (20%)

  By Pattern Type:
    SELECT_STAR: 6
    FULL_TABLE_SCAN: 1

  By Severity:
    MEDIUM: 6
    HIGH: 1


In [9]:
print("\n🎯 Query History Exploration Complete!")
print(f"Space: {TARGET_SPACE_ID}")
print(f"Unique queries: {len(enriched)}")
print(f"Queries needing optimization: {queries_with_issues}")


🎯 Query History Exploration Complete!
Space: 01f10e753b941c87ac6dd970a5c360f1
Unique queries: 35
Queries needing optimization: 7
